In [1]:
from datetime import datetime

# Obtém a data e horário atual
agora = datetime.now()

# Exibe formatado
print("Data e horário atual:", agora.strftime("%d/%m/%Y %H:%M:%S"))

Data e horário atual: 24/09/2026 09:42:31


In [2]:
"""
e1_multistep_llm_finqa_autogen.py

VERSAO 2 (apos a analise dos resultados do E2) -- mudancas que valem para
TODOS os experimentos E0..E7 (rode de novo E0 e E1 para manter a comparacao):
    - HISTORY: o historico da conversa (perguntas/respostas anteriores do
      ConvFinQA) agora vai no bloco HISTORY. Antes ia misturado no TEXT e
      HISTORY aparecia como "(none)" em 64 das 100 perguntas.
    - Leitura de JSON: pega o ULTIMO objeto JSON completo da resposta (antes
      juntava varios blocos num JSON invalido).
    - Nova metrica EM_escala_percentual: no ConvFinQA os percentuais estao
      como fracao (36.93% = 0.3693). Em perguntas de percentual/razao ela
      aceita tambem gold x100 ou gold/100. Use-a para comparar E0..E7; o EM
      estrito continua sendo reportado. O CSV ganhou as colunas question,
      is_percent_question e correct_escala_pct.


EXPERIMENTO E1 - MULTI-STEP da ablacao.

Pergunta cientifica: a decomposicao explicita do problema melhora o
desempenho em relacao a resposta direta (E0)?

Versao LOCAL (via Ollama) com UMA UNICA chamada ao LLM por pergunta.
Nenhum critico, nenhum PoT, nenhuma calculadora: o proprio LLM extrai,
raciocina e calcula -- a unica diferenca para o E0 e que agora ele e
instruido a fazer isso em etapas e devolver uma saida estruturada.

Fluxo (por pergunta):

    Contexto (Question + Text + Table + History)   <- MESMA entrada do E0
        |
        v
       LLM   (system: mesma frase de persona do E0;
        |      user: prompt E1 do documento de ablacao)
        v
    Extracao + raciocinio + calculo (JSON)
        |
        v
    Resposta

Mantido IDENTICO ao E0 (so muda o componente testado):
    - AutoGen (pyautogen==0.2.35), modelo (llama3:8b), temperatura (0.2),
      CSV/amostra, MAX_ITEMS, 1 chamada por pergunta
    - system prompt (frase de persona do script de referencia)
    - montagem da entrada (build_fields: QUESTION / TEXT / TABLE / HISTORY)
    - as duas primeiras frases do prompt E0 ("You are a financial question
      answering assistant." / "Use the provided financial context ...")
    - criterio de acerto (_answers_match, tol = 1e-2)
    - todas as metricas do E0, com as mesmas formulas e nomes de coluna

O que muda (o componente testado):
    - a linha "Return only the final answer." do E0 e substituida pelas
      instrucoes em etapas e pelo formato JSON do prompt E1 do documento:
      evidence / operation / reasoning_summary / answer / unit.

Leitura da resposta:
    - o JSON e lido por _parse_json_response (versao 2: ultimo objeto JSON
      completo da resposta);
    - se "answer" vier como texto (ex.: "25%", "$1.2 million"), o numero e
      limpo com extract_direct_answer() (a mesma funcao do E0);
    - se o JSON for invalido, extract_direct_answer() e aplicada ao texto
      bruto (mesmo tratamento do E0). O modo usado fica em answer_extraction.

Metricas (as mesmas do E0, para comparacao direta):
    - Answer-level: EM, MAE, erro relativo medio
    - MSE / RMSE
    - Intervalo de Confianca de Wilson (95%) para o EM
    - Unit accuracy heuristica
    - Operacionais: tokens, custo hipotetico, latencia, chamadas ao LLM,
      respostas sem numero extraivel

Metricas NOVAS, que passam a fazer sentido porque agora ha estrutura:
    - Conformidade de formato: % de JSON valido e % de respostas com cada
      campo preenchido (evidence, operation, reasoning_summary, unit).
    - [OPCIONAL, desligada por padrao] Avaliacao reasoning-level por LLM
      (Extraction / Plan / Execution Accuracy). Agora ha raciocinio visivel,
      entao o avaliador recebe evidence, operation e reasoning_summary --
      nao so a resposta final, como no script de referencia. Custa 1
      chamada extra por pergunta; os tokens do avaliador NAO entram no
      custo do experimento.

Comparacoes entre experimentos (Kappa, McNemar) serao feitas depois, num
script separado, entre os CSVs de E0..E7 pareados por question_id, usando
a coluna 'correct' de cada um.

Saidas:
    - OUT_PATH                        -> resultados por pergunta (CSV)
    - *_metricas.csv                  -> metricas agregadas (1 linha)
    - *_per_example.jsonl             -> {"id","gold","prediction","correct",
                                          "latency","llm_calls"} por exemplo
    - *_reasoning_level.csv           -> so se RUN_REASONING_LEVEL_EVAL=True

Como rodar localmente:
    1. Instale e rode o Ollama: https://ollama.com
    2. Baixe o modelo:            ollama pull llama3:8b
    3. Confirme que o servidor esta ativo em http://localhost:11434
    4. Instale as dependencias:   pip install "pyautogen==0.2.35" pandas
       (IMPORTANTE: a partir da versao 0.4/pyautogen>=0.9 o pacote virou um
       proxy para a nova arquitetura assincrona "autogen-agentchat". Este
       script usa a API classica sincrona, por isso a versao fixada 0.2.35.)
    5. Ajuste as variaveis de configuracao abaixo se necessario
    6. Rode:                      python e1_multistep_llm_finqa_autogen.py

Requisitos:
    pip install "pyautogen==0.2.35" pandas
    (opcional) pip install scipy   -> z exato para o Intervalo de Confianca
"""

import json
import math
import re
import sys
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import pandas as pd

try:
    import autogen
    from autogen import AssistantAgent
except ImportError:
    print("Instale o SDK primeiro: pip install \"pyautogen==0.2.35\"", file=sys.stderr)
    raise

# scipy e usado APENAS para o valor "z" exato do Intervalo de Confianca.
# E opcional: se nao estiver instalado, o script usa valores de z
# tabelados (90/95/99%).
try:
    from scipy import stats as scipy_stats
    SCIPY_AVAILABLE = True
except ImportError:
    scipy_stats = None
    SCIPY_AVAILABLE = False


# ---------------------------------------------------------------------------
# CONFIGURACAO - ajuste aqui
# ---------------------------------------------------------------------------

EXPERIMENT_ID = "E1_multistep"
CSV_PATH = "convfinqa_amostra_100.csv"
OLLAMA_MODEL = "llama3:8b"
OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_API_KEY = "ollama"          # Ollama ignora o valor, mas o SDK exige algum
MAX_ITEMS = 100                    # quantas linhas do CSV serao lidas/processadas
TIMEOUT_SECONDS = 600              # timeout por chamada ao modelo
OUT_PATH = "results_E1_multistep_autogen_v2.csv"
PER_EXAMPLE_JSONL_PATH = OUT_PATH.replace(".csv", "_per_example.jsonl")
VERBOSE = True

# Mesma temperatura do E0 (e do especialista do script de referencia).
LLM_TEMPERATURE = 0.2

# [OPCIONAL] Avaliacao reasoning-level por LLM (Extraction / Plan /
# Execution Accuracy). Custa 1 chamada extra por pergunta. Desligada por
# padrao: o avaliador e o proprio llama3:8b julgando o proprio raciocinio,
# entao trate o resultado como diagnostico, nao como metrica principal.
RUN_REASONING_LEVEL_EVAL = False
EVALUATOR_MODEL = OLLAMA_MODEL

# Colunas do CSV. O script de referencia usa apenas "text" (contexto +
# pergunta juntos) e "output" (gabarito). Se o seu CSV tiver colunas
# separadas para pergunta / tabela / historico (ex.: ConvFinQA), informe os
# nomes aqui para preencher cada bloco do prompt E0. Colunas inexistentes
# sao ignoradas automaticamente (fallback descrito em build_fields).
TEXT_COL = "text"
GOLD_COL = "output"
QUESTION_COL = "question"          # opcional
TABLE_COL = "table"                # opcional
HISTORY_COL = "history"            # opcional (ConvFinQA)

# Preco de referencia (USD por 1 milhao de tokens) usado SOMENTE para simular
# quanto custaria caso essa mesma carga de tokens fosse rodada em uma API
# paga. A execucao real deste script e local/gratuita via Ollama.
REFERENCE_API_PRICING = {
    "Groq - Llama 3.1 8B Instant": {"input_per_1m": 0.05, "output_per_1m": 0.08},
    "OpenAI - GPT-4o mini":        {"input_per_1m": 0.15, "output_per_1m": 0.60},
}


# ---------------------------------------------------------------------------
# Prompts
# ---------------------------------------------------------------------------

# (1) System prompt: a MESMA frase de persona usada no E0 (primeira frase
#     do prompt do especialista do script de referencia). Manter igual em
#     E0..E7 para que ela nao vire uma variavel da ablacao.
SYSTEM_PERSONA = """You are a financial analysis agent specializing in interpreting \
earnings reports and financial statements. Your task is to answer specific \
financial questions based on the given context from financial reports."""

# (2) Prompt do Experimento E1.
#     - As duas primeiras frases e os blocos QUESTION/TEXT/TABLE/HISTORY sao
#       IDENTICOS ao E0 (mesma entrada).
#     - A linha "Return only the final answer." do E0 foi substituida pelo
#       prompt E1 do documento de ablacao (etapas + JSON), mantido
#       integralmente. As chaves do JSON estao duplicadas ({{ }}) apenas
#       por causa do str.format().
E1_PROMPT_TEMPLATE = """You are a financial question answering assistant.

Use the provided financial context to answer the question.

Analyze the financial question in stages:
1. Identify the relevant values.
2. Identify what the question is asking.
3. Determine the required calculation.
4. Compute the result.
5. Return the final answer.

Return JSON:
{{
  "evidence": [],
  "operation": "",
  "reasoning_summary": "",
  "answer": null,
  "unit": ""
}}

QUESTION:
{question}

TEXT:
{text}

TABLE:
{table}

HISTORY:
{history}
"""


# ---------------------------------------------------------------------------
# Estruturas de dados
# ---------------------------------------------------------------------------

@dataclass
class MultiStepResult:
    question_id: object
    ground_truth: str

    # resposta final (campo "answer" do JSON, normalizado)
    prediction: str = ""
    correct: bool = False
    raw_response: str = ""
    # como a resposta foi lida: json | numero_puro | apos_answer |
    # percentual | ultimo_numero_nao_ano | ultimo_numero | nenhum
    answer_extraction: str = ""
    parse_error: bool = False   # True se nenhum numero foi encontrado

    # saida estruturada do E1
    json_valid: bool = False
    evidence: str = ""          # lista "evidence" serializada em JSON
    operation: str = ""
    reasoning_summary: str = ""
    unit_declared: str = ""

    # tokens / custo operacional desta linha
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_tokens: int = 0
    latency_seconds: float = 0.0
    llm_calls: int = 0


# ---------------------------------------------------------------------------
# Utilitarios de parsing / comparacao de respostas
# ---------------------------------------------------------------------------

def _parse_json_response(text: str) -> dict:
    """Extrai o ULTIMO objeto JSON completo e valido da resposta do modelo.

    Os modelos pequenos costumam escrever texto com varios blocos JSON no
    meio (ex.: resultados parciais e depois o final). A versao anterior
    pegava tudo entre o primeiro "{" e o ultimo "}", o que juntava varios
    blocos num JSON invalido. Agora cada objeto de nivel superior e
    delimitado pelo balanceamento das chaves (ignorando chaves dentro de
    strings) e o ultimo que for valido e devolvido."""
    if text is None:
        return {"raw": "", "parse_error": True}
    candidates = []
    depth, start, in_str, esc = 0, None, False, False
    for i, ch in enumerate(text):
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        if ch == '"' and depth > 0:
            in_str = True
        elif ch == "{":
            if depth == 0:
                start = i
            depth += 1
        elif ch == "}" and depth > 0:
            depth -= 1
            if depth == 0 and start is not None:
                candidates.append(text[start:i + 1])
                start = None
    for cand in reversed(candidates):
        for attempt in (cand, cand.replace("'", '"')):
            try:
                obj = json.loads(attempt)
            except json.JSONDecodeError:
                continue
            if isinstance(obj, dict):
                return obj
    return {"raw": text, "parse_error": True}


def _extract_content(reply) -> str:
    """Extrai com seguranca o texto de uma resposta do AutoGen, que pode vir
    como str, dict ({'content': ...}) ou None (timeout/erro de conexao com
    o Ollama). Retorna sempre uma string (vazia se nao houver conteudo)."""
    if reply is None:
        return ""
    if isinstance(reply, str):
        return reply
    if isinstance(reply, dict):
        return reply.get("content", "") or ""
    return str(reply)


def _to_float(x):
    if x is None:
        return None
    s = str(x).strip().replace(",", "").replace("%", "").replace("$", "")
    try:
        return float(s)
    except ValueError:
        return None


def _answers_match(a, b, tol: float = 1e-2) -> bool:
    fa, fb = _to_float(a), _to_float(b)
    if fa is None or fb is None:
        return str(a).strip() == str(b).strip()
    return abs(fa - fb) <= tol * max(1.0, abs(fb))


# ---------------------------------------------------------------------------
# METRICAS
# ---------------------------------------------------------------------------
# Operam em cima do DataFrame de resultados (out_df) depois que todas as
# linhas foram processadas. Nao fazem nenhuma chamada ao LLM.
# Colunas usadas: prediction, correct, ground_truth.

def _pair_valid_floats(pred_series: pd.Series, gt_series: pd.Series):
    """Devolve duas listas (pred, gt) so com pares onde AMBOS os valores sao
    numericos validos. Usado para MAE / erro relativo / MSE, que nao fazem
    sentido para respostas nao-numericas ou nao-parseaveis."""
    preds, gts = [], []
    for p, g in zip(pred_series, gt_series):
        fp, fg = _to_float(p), _to_float(g)
        if fp is not None and fg is not None:
            preds.append(fp)
            gts.append(fg)
    return preds, gts


# ---------------------------------------------------------------------------
# EM com tolerancia de escala percentual
# ---------------------------------------------------------------------------
# No ConvFinQA os percentuais estao no gabarito como FRACAO (36.93% = 0.3693),
# mas alguns poucos estao em pontos percentuais. Um modelo que responde
# "36.93" para uma pergunta de percentual acerta o conteudo e erra so a
# convencao. O EM estrito continua sendo o criterio principal; o
# EM_escala_percentual aceita, SOMENTE em perguntas de percentual/razao,
# tambem prediction == gold*100 ou prediction == gold/100. E a metrica
# recomendada para comparar E0..E7, porque nao depende de cada prompt
# conhecer a convencao do dataset.

_PERCENT_Q_RE = re.compile(
    r"(?i)percent|%|\bportion\b|proportion|\bratio\b|represent|return rate|growth rate")


def is_percent_question(question: str) -> bool:
    return bool(_PERCENT_Q_RE.search(str(question or "")))


def answers_match_pct_scale(pred, gold, question: str) -> bool:
    if _answers_match(pred, gold):
        return True
    if not is_percent_question(question):
        return False
    fp, fg = _to_float(pred), _to_float(gold)
    if fp is None or fg is None:
        return False
    return _answers_match(fp, fg * 100) or _answers_match(fp, fg / 100)


def add_question_columns(out_df: pd.DataFrame, subset_df: pd.DataFrame) -> pd.DataFrame:
    """Acrescenta ao CSV a pergunta respondida, se ela e de percentual e o
    acerto com tolerancia de escala percentual."""
    if len(out_df) == 0:
        return out_df
    qs = [build_fields(subset_df.loc[qid])["question"] if qid in subset_df.index else ""
          for qid in out_df["question_id"]]
    out_df.insert(out_df.columns.get_loc("ground_truth"), "question", qs)
    out_df["is_percent_question"] = [is_percent_question(q) for q in qs]
    out_df["correct_escala_pct"] = [answers_match_pct_scale(p, g, q) for p, g, q in
                                    zip(out_df["prediction"], out_df["ground_truth"], qs)]
    return out_df


def compute_answer_level_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """Exact Match / Accuracy, MAE e erro relativo medio da resposta direta,
    comparada ao gabarito (ground_truth)."""
    metrics: Dict[str, float] = {}
    n = len(out_df)
    if n == 0:
        return metrics

    metrics["EM"] = 100.0 * out_df["correct"].sum() / n
    if "correct_escala_pct" in out_df.columns:
        metrics["EM_escala_percentual"] = 100.0 * out_df["correct_escala_pct"].sum() / n
        metrics["n_perguntas_percentual"] = int(out_df["is_percent_question"].sum())

    preds, gts = _pair_valid_floats(out_df["prediction"], out_df["ground_truth"])
    if preds:
        abs_errors = [abs(p - g) for p, g in zip(preds, gts)]
        rel_errors = [abs(p - g) / max(1.0, abs(g)) for p, g in zip(preds, gts)]
        metrics["MAE"] = sum(abs_errors) / len(abs_errors)
        metrics["ErroRelativoMedio"] = 100.0 * sum(rel_errors) / len(rel_errors)
    else:
        metrics["MAE"] = float("nan")
        metrics["ErroRelativoMedio"] = float("nan")
    metrics["n_pares_numericos"] = len(preds)
    return metrics


def compute_squared_error_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """MSE (Erro Quadratico Medio) e RMSE (sua raiz) da resposta direta.

    Complementa o MAE: o MSE eleva os erros ao quadrado antes de tirar a
    media, entao penaliza mais fortemente respostas que erram "feio" (ex.:
    o modelo erra a ordem de grandeza), enquanto o MAE trata todo desvio
    de forma linear."""
    preds, gts = _pair_valid_floats(out_df["prediction"], out_df["ground_truth"])
    if not preds:
        return {"MSE": float("nan"), "RMSE": float("nan"), "n_pares_numericos_mse": 0}
    mse = sum((p - g) ** 2 for p, g in zip(preds, gts)) / len(preds)
    return {"MSE": mse, "RMSE": math.sqrt(mse), "n_pares_numericos_mse": len(preds)}


def _z_for_confidence(confidence: float) -> float:
    """Valor critico z (distribuicao normal padrao, bicaudal) para o nivel
    de confianca pedido. Usa scipy quando disponivel (qualquer confianca);
    caso contrario, cai para uma tabela com os niveis mais comuns."""
    if SCIPY_AVAILABLE:
        return float(scipy_stats.norm.ppf(1 - (1 - confidence) / 2))
    z_table = {0.90: 1.6449, 0.95: 1.9600, 0.99: 2.5758}
    return z_table.get(round(confidence, 2), 1.9600)  # default: 95%


def _wilson_confidence_interval(successes: int, n: int, confidence: float = 0.95) -> Tuple[float, float]:
    """Intervalo de Confianca de Wilson para uma proporcao (EM/Accuracy).

    Preferido ao intervalo normal/Wald classico porque continua confiavel
    com amostras pequenas (ex.: n=100) e quando a proporcao observada esta
    perto de 0% ou 100% -- casos em que o intervalo de Wald pode extrapolar
    para fora de [0%, 100%]."""
    if n == 0:
        return float("nan"), float("nan")
    z = _z_for_confidence(confidence)
    p = successes / n
    denom = 1 + (z ** 2) / n
    centre = p + (z ** 2) / (2 * n)
    margin = z * math.sqrt((p * (1 - p) / n) + (z ** 2) / (4 * n ** 2))
    low = (centre - margin) / denom
    high = (centre + margin) / denom
    return max(0.0, low), min(1.0, high)


def compute_confidence_interval(out_df: pd.DataFrame, confidence: float = 0.95) -> Dict[str, float]:
    """Intervalo de Confianca (Wilson, 95% por padrao) para o EM.

    Com amostras pequenas (ex.: n=100), duas taxas de EM proximas (ex.:
    48% vs 45% entre E0 e E1) podem nao ser estatisticamente diferentes --
    o IC ajuda a visualizar essa incerteza."""
    n = len(out_df)
    if n == 0:
        return {}
    low, high = _wilson_confidence_interval(int(out_df["correct"].sum()), n, confidence)
    metrics = {
        "EM_IC_low": 100.0 * low,
        "EM_IC_high": 100.0 * high,
        "EM_IC_margem": 100.0 * (high - low) / 2,
        "IC_nivel_confianca": confidence,
        "IC_n_amostras": n,
    }
    if "correct_escala_pct" in out_df.columns:
        low, high = _wilson_confidence_interval(int(out_df["correct_escala_pct"].sum()), n, confidence)
        metrics["EM_escala_pct_IC_low"] = 100.0 * low
        metrics["EM_escala_pct_IC_high"] = 100.0 * high
    return metrics


def compute_unit_heuristic_accuracy(subset_df: pd.DataFrame, out_df: pd.DataFrame) -> Dict[str, float]:
    """Aproximacao HEURISTICA de Unit Accuracy (nao usa LLM): para perguntas
    cujo enunciado sugere uma unidade (%, milhao, bilhao), verifica se a
    resposta esta na mesma ordem de grandeza do gabarito (desvio relativo
    < 50%). Um erro de unidade (ex.: 0.25 em vez de 25%) cai fora dessa
    faixa. E um proxy grosseiro.

    As linhas sao pareadas por question_id (indice do CSV), entao linhas
    que falharam durante a execucao nao desalinham a comparacao."""
    def unit_hint(text: str) -> Optional[str]:
        t = str(text).lower()
        if "%" in t or "percent" in t:
            return "percent"
        if "billion" in t:
            return "billion"
        if "million" in t:
            return "million"
        return None

    matches = 0
    total = 0
    for _, row in out_df.iterrows():
        qid = row["question_id"]
        if qid not in subset_df.index:
            continue
        if unit_hint(subset_df.at[qid, TEXT_COL]) is None:
            continue
        total += 1
        fa, fg = _to_float(row["prediction"]), _to_float(row["ground_truth"])
        if fa is None or fg is None:
            continue
        if fg != 0 and abs(fa - fg) / max(1.0, abs(fg)) < 0.5:
            matches += 1
    return {
        "unit_accuracy_heuristica": 100.0 * matches / total if total else float("nan"),
        "n_perguntas_com_unidade_detectada": total,
    }


def compute_format_metrics(out_df: pd.DataFrame) -> Dict[str, float]:
    """Conformidade com o formato estruturado pedido no prompt E1: % de
    respostas com JSON valido e % com cada campo preenchido. Ajuda a
    separar "o modelo errou o raciocinio" de "o modelo nao seguiu o
    formato"."""
    n = len(out_df)
    if n == 0:
        return {}
    metrics = {"json_valido_pct": 100.0 * out_df["json_valid"].sum() / n}
    for col in ("evidence", "operation", "reasoning_summary", "unit_declared"):
        filled = out_df[col].apply(lambda v: _is_filled(_parse_maybe_json(v)))
        metrics[f"{col}_preenchido_pct"] = 100.0 * filled.sum() / n
    return metrics


def _parse_maybe_json(v):
    if not isinstance(v, str):
        return v
    s = v.strip()
    if s.startswith("[") or s.startswith("{"):
        try:
            return json.loads(s)
        except json.JSONDecodeError:
            return s
    return s


# ---------------------------------------------------------------------------
# Configuracao AutoGen
# ---------------------------------------------------------------------------

def _base_llm_config(temperature: float, model: str = OLLAMA_MODEL) -> dict:
    return {
        "config_list": [{
            "model": model,
            "base_url": OLLAMA_BASE_URL,
            "api_key": OLLAMA_API_KEY,
        }],
        "temperature": temperature,
        "timeout": TIMEOUT_SECONDS,
        "cache_seed": None,  # desliga cache do autogen p/ nao mascarar tokens/custos
    }



def build_agents():
    agent = AssistantAgent(
        name="MultiStep_LLM",
        system_message=SYSTEM_PERSONA,
        llm_config=_base_llm_config(LLM_TEMPERATURE),
        human_input_mode="NEVER",
        max_consecutive_auto_reply=1,
        code_execution_config=False,
    )
    return agent


def _usage_totals(agents: List) -> Dict[str, int]:
    """Soma prompt/completion/total tokens de uma lista de agentes autogen,
    usando autogen.gather_usage_summary (cobre todos os modelos usados)."""
    summary = autogen.gather_usage_summary(agents)
    totals = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    usage = summary.get("usage_including_cached_inference", {})
    for model_name, stats in usage.items():
        if model_name == "total_cost":
            continue
        totals["prompt_tokens"] += stats.get("prompt_tokens", 0)
        totals["completion_tokens"] += stats.get("completion_tokens", 0)
        totals["total_tokens"] += stats.get("total_tokens", 0)
    return totals




# ---------------------------------------------------------------------------
# Montagem da entrada (QUESTION / TEXT / TABLE / HISTORY) -- identica ao E0
# ---------------------------------------------------------------------------

_FIRST_TURN_RE = re.compile(r"(?i)(?:^|\n)[ \t]*question[ \t]*:")
_TURN_RE = re.compile(
    r"(?is)question[ \t]*:[ \t]*(.*?)(?:\n[ \t]*answer[ \t]*:[ \t]*(.*?))?[ \t]*(?=\n[ \t]*question[ \t]*:|\Z)")


def _cell(row: pd.Series, col: Optional[str]) -> Optional[str]:
    """Le uma coluna opcional do CSV; devolve None se nao existir/vazia."""
    if not col or col not in row.index:
        return None
    val = row[col]
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return None
    s = str(val).strip()
    return s or None


def split_dialogue(full_text: str) -> Tuple[str, List[Tuple[str, str]], Optional[str]]:
    """Separa o texto do ConvFinQA em (contexto, historico, ultima pergunta).

    No CSV, a coluna "text" traz o relatorio (texto + tabela) seguido do
    dialogo: "Question: ... Answer: ... Question: ...". A ultima pergunta
    (sem resposta) e a que deve ser respondida; os pares anteriores sao o
    historico da conversa."""
    m = _FIRST_TURN_RE.search(full_text)
    if not m:
        return full_text.strip(), [], None
    context = full_text[:m.start()].strip()
    turns = [(q.strip(), (a or "").strip()) for q, a in _TURN_RE.findall(full_text[m.start():])]
    turns = [t for t in turns if t[0]]
    if not turns:
        return context, [], None
    return context, turns[:-1], turns[-1][0]


def build_fields(row: pd.Series) -> Dict[str, str]:
    """Preenche os blocos QUESTION / TEXT / TABLE / HISTORY (igual em E0..E7).

    - Colunas separadas (QUESTION_COL, TABLE_COL, HISTORY_COL), se existirem
      no CSV, sao usadas diretamente.
    - Caso contrario, a coluna "text" e dividida por split_dialogue():
      TEXT = relatorio (texto + tabela), QUESTION = ultima pergunta do
      dialogo e HISTORY = perguntas e respostas anteriores. Na versao
      anterior o historico ia misturado no TEXT e HISTORY aparecia como
      "(none)" mesmo em conversas de varios turnos.
    - Se nao houver "Question:" no texto, tudo vai para TEXT e QUESTION
      indica que a pergunta esta nele. Nenhuma informacao e descartada.
    """
    full_text = _cell(row, TEXT_COL) or ""
    question = _cell(row, QUESTION_COL)
    table = _cell(row, TABLE_COL)
    history = _cell(row, HISTORY_COL)

    text = full_text
    if question is None:
        context, turns, last_q = split_dialogue(full_text)
        if last_q:
            text, question = context, last_q
            if history is None and turns:
                history = "\n".join(f"Question: {q}\nAnswer: {a}" for q, a in turns)
        else:
            question = "(The question is included at the end of the TEXT block.)"

    return {
        "question": question,
        "text": text if text else "(not provided)",
        "table": table if table else "(not provided separately -- any table is included in the TEXT block)",
        "history": history if history else "(none)",
    }


def build_e1_prompt(row: pd.Series) -> str:
    return E1_PROMPT_TEMPLATE.format(**build_fields(row))


_NUM_RE = re.compile(r"-?\$?\d[\d,]*(?:\.\d+)?%?")


def _clean_num(tok: str) -> Optional[str]:
    t = tok.replace("$", "").replace(",", "").replace("%", "").strip()
    return t if _to_float(t) is not None else None


def _is_year_like(tok: str) -> bool:
    t = tok.replace("$", "").replace(",", "")
    return bool(re.fullmatch(r"(19|20)\d{2}", t))


def extract_direct_answer(raw: str) -> Tuple[Optional[str], str]:
    """Extrai um numero de um texto livre (MESMA funcao do E0). No E1 e
    usada para limpar um "answer" textual (ex.: "25%") ou, se o JSON vier
    invalido, para ler o texto bruto. Ordem de leitura:
      1. JSON com campo "answer" (se o modelo produzir JSON por conta propria)
      2. resposta inteira e so um numero (ex.: "25", "25%", "$1,200")
      3. numero logo apos "answer is" / "answer:" / "="
      4. ultimo numero com "%" (perguntas de percentual)
      5. ultimo numero que nao parece um ano (1900-2099)
      6. ultimo numero qualquer
    Devolve (resposta, modo_de_extracao)."""
    text = (raw or "").strip()
    if not text:
        return None, "nenhum"

    parsed = _parse_json_response(text)
    if not parsed.get("parse_error") and parsed.get("answer") is not None:
        return str(parsed.get("answer")), "json"

    stripped = re.sub(r"(?i)\b(million|billion|thousand|percent)\b", "", text).strip(" .\n\t*")
    if _clean_num(stripped) is not None:
        return _clean_num(stripped), "numero_puro"

    m = re.search(r"(?i)(?:final answer|answer)\s*(?:is|:|=)?\s*[:=]?\s*(-?\$?\d[\d,]*(?:\.\d+)?%?)", text)
    if m and _clean_num(m.group(1)) is not None:
        return _clean_num(m.group(1)), "apos_answer"

    tokens = _NUM_RE.findall(text)
    pct = [t for t in tokens if t.endswith("%")]
    if pct:
        return _clean_num(pct[-1]), "percentual"
    non_year = [t for t in tokens if not _is_year_like(t)]
    if non_year:
        return _clean_num(non_year[-1]), "ultimo_numero_nao_ano"
    if tokens:
        return _clean_num(tokens[-1]), "ultimo_numero"
    return None, "nenhum"


def extract_multistep_answer(raw: str) -> Tuple[dict, Optional[str], str]:
    """Le a resposta do E1. Devolve (json_parseado, resposta, modo).

    Modos:
      json                 -> "answer" do JSON ja e um numero
      json_texto_limpo     -> "answer" veio como texto (ex.: "25%") e foi
                              limpo com extract_direct_answer
      json_sem_answer      -> JSON valido, mas "answer" nulo/vazio/sem numero
      fallback_<modo_E0>   -> JSON invalido; extract_direct_answer no texto bruto
    """
    parsed = _parse_json_response(raw)
    if not parsed.get("parse_error"):
        ans = parsed.get("answer")
        if ans is not None and str(ans).strip() != "":
            if _to_float(ans) is not None:
                return parsed, str(_to_float(ans)) if isinstance(ans, str) else str(ans), "json"
            cleaned, _ = extract_direct_answer(str(ans))
            if cleaned is not None:
                return parsed, cleaned, "json_texto_limpo"
        return parsed, None, "json_sem_answer"

    answer, mode = extract_direct_answer(raw)
    return parsed, answer, f"fallback_{mode}"


def _as_text(x) -> str:
    if x is None:
        return ""
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    return str(x)


def _is_filled(x) -> bool:
    if x is None:
        return False
    if isinstance(x, (list, dict)):
        return len(x) > 0
    return str(x).strip() not in ("", "null", "None")


# ---------------------------------------------------------------------------
# Fluxo E1: Contexto -> LLM (em etapas, JSON) -> Resposta
# (1 unica chamada, sem critico, sem PoT, sem calculadora)
# ---------------------------------------------------------------------------

def run_multistep(agent, row: pd.Series, ground_truth: str,
                  verbose: bool = False) -> MultiStepResult:
    prompt = build_e1_prompt(row)

    t0 = time.perf_counter()
    reply = agent.generate_reply(messages=[{"role": "user", "content": prompt}])
    latency = time.perf_counter() - t0

    raw = _extract_content(reply)
    parsed, answer, mode = extract_multistep_answer(raw)
    json_valid = not parsed.get("parse_error", False)
    prediction = str(answer)

    if verbose:
        print(f"    [{agent.name}] -> {prediction}  (extracao: {mode}) | "
              f"operation: {_as_text(parsed.get('operation'))[:80]}")

    return MultiStepResult(
        question_id=None,
        ground_truth=str(ground_truth),
        prediction=prediction,
        correct=_answers_match(prediction, ground_truth),
        raw_response=raw,
        answer_extraction=mode,
        parse_error=answer is None,
        json_valid=json_valid,
        evidence=_as_text(parsed.get("evidence")) if json_valid else "",
        operation=_as_text(parsed.get("operation")) if json_valid else "",
        reasoning_summary=_as_text(parsed.get("reasoning_summary")) if json_valid else "",
        unit_declared=_as_text(parsed.get("unit")) if json_valid else "",
        latency_seconds=latency,
        llm_calls=1,
    )


# ---------------------------------------------------------------------------
# [OPCIONAL] Avaliacao reasoning-level por LLM
# ---------------------------------------------------------------------------

EVALUATOR_SYSTEM = """You are an evaluation agent. You will be given a financial \
context, a question, a ground-truth final answer, and a candidate answer with its \
reasoning (extracted evidence, operation and reasoning summary). Judge THREE separate \
things and respond ONLY with a JSON object, no extra text:
{
  "extraction_accuracy": 0 or 1,   // did the candidate correctly identify/extract the \
relevant numbers from the context?
  "plan_accuracy": 0 or 1,         // is the sequence of operations/logic (the plan) \
correct, regardless of arithmetic slips?
  "execution_accuracy": 0 or 1     // was the arithmetic/execution of that plan carried \
out correctly?
}
Base your judgment on the context and the ground-truth answer provided, not on your own \
independent recomputation."""


def run_reasoning_level_judge(out_df: pd.DataFrame, subset_df: pd.DataFrame) -> pd.DataFrame:
    """Avalia Extraction / Plan / Execution Accuracy com o MESMO prompt de
    avaliador do script de referencia, mas agora passando o raciocinio
    visivel do E1 (evidence, operation, reasoning_summary) -- no script de
    referencia o avaliador recebia apenas a resposta final.
    Linhas com JSON invalido nao tem raciocinio a avaliar e sao puladas."""
    evaluator = AssistantAgent(
        name="Evaluator",
        system_message=EVALUATOR_SYSTEM,
        llm_config=_base_llm_config(0.0, model=EVALUATOR_MODEL),
        human_input_mode="NEVER",
        max_consecutive_auto_reply=1,
        code_execution_config=False,
    )

    results = []
    for _, row in out_df.iterrows():
        qid = row["question_id"]
        if not row["json_valid"] or qid not in subset_df.index:
            continue
        fields = build_fields(subset_df.loc[qid])
        prompt = (
            f"QUESTION:\n{fields['question']}\n\n"
            f"TEXT:\n{fields['text']}\n\n"
            f"TABLE:\n{fields['table']}\n\n"
            f"HISTORY:\n{fields['history']}\n\n"
            f"Ground-truth final answer: {row['ground_truth']}\n\n"
            f"Candidate evidence: {row['evidence']}\n"
            f"Candidate operation: {row['operation']}\n"
            f"Candidate reasoning summary: {row['reasoning_summary']}\n"
            f"Candidate final answer: {row['prediction']} {row['unit_declared']}\n"
        )
        try:
            reply = evaluator.generate_reply(messages=[{"role": "user", "content": prompt}])
            parsed = _parse_json_response(_extract_content(reply))
        except Exception as e:
            print(f"  [reasoning-level] ERRO na linha question_id={qid}: {e}", file=sys.stderr)
            parsed = {}
        results.append({
            "question_id": qid,
            "extraction_accuracy": parsed.get("extraction_accuracy"),
            "plan_accuracy": parsed.get("plan_accuracy"),
            "execution_accuracy": parsed.get("execution_accuracy"),
        })
    return pd.DataFrame(results, columns=["question_id", "extraction_accuracy",
                                          "plan_accuracy", "execution_accuracy"])


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def _fmt(x, spec=".2f"):
    try:
        return "N/D" if x is None or (isinstance(x, float) and math.isnan(x)) else format(x, spec)
    except (TypeError, ValueError):
        return str(x)


def main():
    df = pd.read_csv(CSV_PATH)
    subset = df.head(MAX_ITEMS)

    results: List[MultiStepResult] = []
    n_correct = 0

    print(f"Experimento: {EXPERIMENT_ID}")
    print(f"Modelo: {OLLAMA_MODEL} | Endpoint: {OLLAMA_BASE_URL}")
    print(f"Processando {len(subset)} de {len(df)} linhas do CSV (MAX_ITEMS={MAX_ITEMS})")
    print(f"LLM multi-step (temperatura {LLM_TEMPERATURE}) | 1 chamada por pergunta | "
          "sem critico, sem PoT, sem calculadora\n")

    for i, (idx, row) in enumerate(subset.iterrows(), start=1):
        ground_truth = row[GOLD_COL]

        print(f"[{i}/{len(subset)}] Processando linha id={idx} ...")

        agent = build_agents()
        usage_before = _usage_totals([agent])

        try:
            result = run_multistep(agent, row, ground_truth, verbose=VERBOSE)
        except Exception as e:
            print(f"  ERRO: {e}", file=sys.stderr)
            continue

        usage_after = _usage_totals([agent])
        result.question_id = idx
        result.prompt_tokens = usage_after["prompt_tokens"] - usage_before["prompt_tokens"]
        result.completion_tokens = usage_after["completion_tokens"] - usage_before["completion_tokens"]
        result.total_tokens = usage_after["total_tokens"] - usage_before["total_tokens"]

        results.append(result)
        n_correct += int(result.correct)

        print(f"  -> prediction: {result.prediction} (correto: {result.correct}) | "
              f"gt: {result.ground_truth} | tokens: {result.total_tokens} | "
              f"latencia: {result.latency_seconds:.2f}s\n")

        time.sleep(0.1)

    # ------------------------------------------------------------------
    # Salva resultados detalhados (1 linha por pergunta)
    # ------------------------------------------------------------------
    out_df = pd.DataFrame([{
        "experiment": EXPERIMENT_ID,
        "question_id": r.question_id,
        "ground_truth": r.ground_truth,
        "prediction": r.prediction,
        "correct": r.correct,
        "answer_extraction": r.answer_extraction,
        "parse_error": r.parse_error,
        "json_valid": r.json_valid,
        "evidence": r.evidence,
        "operation": r.operation,
        "reasoning_summary": r.reasoning_summary,
        "unit_declared": r.unit_declared,
        "raw_response": r.raw_response,
        "prompt_tokens": r.prompt_tokens,
        "completion_tokens": r.completion_tokens,
        "total_tokens": r.total_tokens,
        "latency_seconds": r.latency_seconds,
        "llm_calls": r.llm_calls,
    } for r in results])
    out_df = add_question_columns(out_df, subset)
    out_df.to_csv(OUT_PATH, index=False)

    # Registro por exemplo no formato pedido no documento de ablacao.
    with open(PER_EXAMPLE_JSONL_PATH, "w", encoding="utf-8") as f:
        for r in results:
            gold_f, pred_f = _to_float(r.ground_truth), _to_float(r.prediction)
            f.write(json.dumps({
                "id": str(r.question_id),
                "gold": gold_f if gold_f is not None else r.ground_truth,
                "prediction": pred_f if pred_f is not None else (None if r.prediction == "None" else r.prediction),
                "correct": bool(r.correct),
                "latency": round(r.latency_seconds, 3),
                "llm_calls": r.llm_calls,
            }, ensure_ascii=False) + "\n")

    n = len(results)

    print("=" * 70)
    print(f"RESUMO - {EXPERIMENT_ID}: LLM multi-step (AutoGen)")
    print("=" * 70)
    if n == 0:
        print("Nenhuma linha processada com sucesso.")
        return

    total_prompt_tokens = sum(r.prompt_tokens for r in results)
    total_completion_tokens = sum(r.completion_tokens for r in results)
    total_tokens = sum(r.total_tokens for r in results)
    n_parse_errors = sum(int(r.parse_error) for r in results)
    total_latency = sum(r.latency_seconds for r in results)
    total_calls = sum(r.llm_calls for r in results)

    print(f"Linhas avaliadas:                          {n}")
    print(f"Respostas sem numero extraivel:            {n_parse_errors}/{n} "
          f"({100 * n_parse_errors / n:.2f}%)")
    print("Modo de extracao da resposta:")
    for mode, cnt in out_df["answer_extraction"].value_counts().items():
        print(f"    {mode:<24} {cnt}")

    # ------------------------------------------------------------------
    # Metricas
    # ------------------------------------------------------------------
    answer_metrics = compute_answer_level_metrics(out_df)
    squared_error_metrics = compute_squared_error_metrics(out_df)
    ci_metrics = compute_confidence_interval(out_df, confidence=0.95)
    unit_metrics = compute_unit_heuristic_accuracy(subset, out_df)
    format_metrics = compute_format_metrics(out_df)

    print("\n" + "-" * 70)
    print("ACERTOS")
    print("-" * 70)
    print(f"Acertos: {n_correct}/{n} -> EM = {_fmt(answer_metrics.get('EM'))}%")
    print(f"IC {int(ci_metrics.get('IC_nivel_confianca', 0.95) * 100)}% (Wilson): "
          f"[{_fmt(ci_metrics.get('EM_IC_low'))}%, {_fmt(ci_metrics.get('EM_IC_high'))}%] "
          f"(margem +-{_fmt(ci_metrics.get('EM_IC_margem'))} p.p.)")
    print(f"EM com escala percentual: {_fmt(answer_metrics.get('EM_escala_percentual'))}% "
          f"[{_fmt(ci_metrics.get('EM_escala_pct_IC_low'))}%, {_fmt(ci_metrics.get('EM_escala_pct_IC_high'))}%] "
          f"(aceita gold x100 ou /100 nas {answer_metrics.get('n_perguntas_percentual', 0)} perguntas "
          "de percentual; use esta para comparar E0..E7)")

    print("\n" + "-" * 70)
    print("ERRO NUMERICO (apenas pares com resposta numerica valida)")
    print("-" * 70)
    print(f"MAE:                 {_fmt(answer_metrics.get('MAE'), '.4f')} "
          f"(n={answer_metrics.get('n_pares_numericos', 0)})")
    print(f"Erro relativo medio: {_fmt(answer_metrics.get('ErroRelativoMedio'))}%")
    print(f"MSE:                 {_fmt(squared_error_metrics.get('MSE'), '.4f')}")
    print(f"RMSE:                {_fmt(squared_error_metrics.get('RMSE'), '.4f')}")

    print("\n" + "-" * 70)
    print("UNIDADE (heuristico, sem LLM)")
    print("-" * 70)
    print(f"Unit accuracy heuristica: {_fmt(unit_metrics.get('unit_accuracy_heuristica'))}% "
          f"(n={unit_metrics.get('n_perguntas_com_unidade_detectada', 0)})")

    print("\n" + "-" * 70)
    print("CONFORMIDADE COM O FORMATO ESTRUTURADO (novo no E1)")
    print("-" * 70)
    print(f"JSON valido:                  {_fmt(format_metrics.get('json_valido_pct'))}%")
    print(f"evidence preenchido:          {_fmt(format_metrics.get('evidence_preenchido_pct'))}%")
    print(f"operation preenchido:         {_fmt(format_metrics.get('operation_preenchido_pct'))}%")
    print(f"reasoning_summary preenchido: {_fmt(format_metrics.get('reasoning_summary_preenchido_pct'))}%")
    print(f"unit preenchido:              {_fmt(format_metrics.get('unit_declared_preenchido_pct'))}%")

    print("\n" + "-" * 70)
    print("CUSTO OPERACIONAL")
    print("-" * 70)
    print(f"Chamadas ao LLM:              {total_calls} ({total_calls / n:.2f} por pergunta)")
    print(f"Latencia total:               {total_latency:,.1f}s")
    print(f"Latencia media por pergunta:  {total_latency / n:,.2f}s")
    print(f"Prompt tokens (entrada):      {total_prompt_tokens:,}")
    print(f"Completion tokens (saida):    {total_completion_tokens:,}")
    print(f"Total tokens:                 {total_tokens:,}")
    print(f"Media de tokens por pergunta: {total_tokens / n:,.1f}")

    print("\n" + "-" * 70)
    print("COMPARACAO DE CUSTO HIPOTETICO (execucao real foi LOCAL/GRATUITA via Ollama)")
    print("-" * 70)
    print("Os valores abaixo simulam quanto custaria rodar o MESMO volume de "
          "tokens em uma API paga (precos de referencia, sujeitos a mudanca):\n")
    cost_metrics: Dict[str, float] = {}
    for provider, price in REFERENCE_API_PRICING.items():
        cost_input = (total_prompt_tokens / 1_000_000) * price["input_per_1m"]
        cost_output = (total_completion_tokens / 1_000_000) * price["output_per_1m"]
        cost_total = cost_input + cost_output
        cost_metrics[f"custo_hipotetico_usd__{provider}"] = cost_total
        print(f"  {provider}:")
        print(f"    ${price['input_per_1m']:.3f}/1M input, ${price['output_per_1m']:.3f}/1M output")
        print(f"    Custo estimado: ${cost_input:.4f} (input) + ${cost_output:.4f} (output) "
              f"= ${cost_total:.4f} total | ${cost_total / n:.5f} por pergunta")
    print("\n  Ollama (local, GPU/CPU proprios): $0.00 por chamada de API "
          "(custo real e apenas energia/hardware).")

    operational_metrics = {
        "experiment": EXPERIMENT_ID,
        "modelo": OLLAMA_MODEL,
        "temperatura": LLM_TEMPERATURE,
        "n_avaliadas": n,
        "n_sem_numero_extraivel": n_parse_errors,
        "llm_calls_total": total_calls,
        "llm_calls_por_pergunta": total_calls / n,
        "latencia_total_s": total_latency,
        "latencia_media_s": total_latency / n,
        "prompt_tokens_total": total_prompt_tokens,
        "completion_tokens_total": total_completion_tokens,
        "total_tokens": total_tokens,
        "tokens_medios_por_pergunta": total_tokens / n,
    }

    metrics_path = OUT_PATH.replace(".csv", "_metricas.csv")
    all_metrics = {
        **operational_metrics,
        **answer_metrics,
        **squared_error_metrics,
        **ci_metrics,
        **unit_metrics,
        **format_metrics,
        **cost_metrics,
    }

    if RUN_REASONING_LEVEL_EVAL:
        print("\n" + "-" * 70)
        print("REASONING-LEVEL (avaliador LLM, opcional -- custo NAO incluido acima)")
        print("-" * 70)
        reasoning_df = run_reasoning_level_judge(out_df, subset)
        reasoning_path = OUT_PATH.replace(".csv", "_reasoning_level.csv")
        reasoning_df.to_csv(reasoning_path, index=False)
        for col in ("extraction_accuracy", "plan_accuracy", "execution_accuracy"):
            vals = pd.to_numeric(reasoning_df[col], errors="coerce").dropna()
            pct = 100.0 * vals.mean() if len(vals) else float("nan")
            all_metrics[col] = pct
            all_metrics[f"n_{col}"] = len(vals)
            print(f"  {col}: {_fmt(pct)}% (n={len(vals)})")
        print(f"Resultados reasoning-level salvos em: {reasoning_path}")

    pd.DataFrame([all_metrics]).to_csv(metrics_path, index=False)

    print(f"\nResultados detalhados salvos em: {OUT_PATH}")
    print(f"Metricas agregadas salvas em:    {metrics_path}")
    print(f"Registro por exemplo salvo em:   {PER_EXAMPLE_JSONL_PATH}")


if __name__ == "__main__":
    main()


Experimento: E1_multistep
Modelo: llama3:8b | Endpoint: http://localhost:11434/v1
Processando 100 de 100 linhas do CSV (MAX_ITEMS=100)
LLM multi-step (temperatura 0.2) | 1 chamada por pergunta | sem critico, sem PoT, sem calculadora

[1/100] Processando linha id=0 ...
[autogen.oai.client: 09-24 09:42:50] {744} WARNING - Model llama3:8b is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
    [MultiStep_LLM] -> 93.4  (extracao: json_texto_limpo) | operation: addition
  -> prediction: 93.4 (correto: True) | gt: 93.4 | tokens: 1416 | latencia: 12.75s

[2/100] Processando linha id=1 ...
[autogen.oai.client: 09-24 09:42:56] {744} WARNING - Model llama3:8b is not found. The cost will be 0. In your config_list, add field {"price" : [prompt_price_per_1k, completion_token_price_per_1k]} for customized pricing.
    [MultiStep_LLM] -> 198  (extracao: json) | operation: answer
  -> prediction: 198 

In [3]:
from datetime import datetime

# Obtém a data e horário atual
agora = datetime.now()

# Exibe formatado
print("Data e horário atual:", agora.strftime("%d/%m/%Y %H:%M:%S"))

Data e horário atual: 24/09/2026 09:56:08
